# 📊 01 — Análisis Exploratorio de Datos (EDA)
**Proyecto 01: Reciclaje Inteligente — Clasificación de Desechos con Visión Artificial**  
Grupo #4 · Curso 045 — Inteligencia Artificial · UMG 2026  
Responsable: Marvin Vásquez / Javier Rivera

---
## Objetivos de este notebook
1. Explorar la distribución de clases del dataset
2. Visualizar muestras representativas de cada clase
3. Analizar estadísticas de tamaño de imágenes
4. Detectar posibles problemas: desbalance, imágenes borrosas o mal etiquetadas
5. Documentar hallazgos para justificar decisiones de preprocesamiento

## 0. Instalación y configuración

In [ ]:
# Instalar dependencias si es necesario
# !pip install torch torchvision matplotlib seaborn pillow numpy scikit-learn opencv-python

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter
import cv2

# Agregar src/ al path para importar módulos del proyecto
sys.path.append(os.path.join('..', 'src'))
from model import CLASSES, COLOR_MAP

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Librerías importadas correctamente')
print(f'Clases del proyecto: {CLASSES}')

## 1. Carga del dataset

In [ ]:
DATA_DIR = Path('../data/raw')

# Verificar que el directorio existe
if not DATA_DIR.exists():
    print('❌ No se encontró data/raw/')
    print('   Ejecuta primero: python download_datasets.py')
else:
    print(f'✅ Directorio encontrado: {DATA_DIR.resolve()}')

# Contar imágenes por clase
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

conteo = {}
rutas   = {}

for clase in CLASSES:
    clase_dir = DATA_DIR / clase
    if clase_dir.exists():
        imgs = [p for p in clase_dir.iterdir() if p.suffix in EXTENSIONES]
        conteo[clase] = len(imgs)
        rutas[clase]  = imgs
    else:
        conteo[clase] = 0
        rutas[clase]  = []
        print(f'  ⚠️  Carpeta no encontrada: {clase_dir}')

total = sum(conteo.values())
print(f'\n📦 Total de imágenes en el dataset: {total}')
print('\nDistribución por clase:')
for clase, n in conteo.items():
    estado = '✅' if n >= 50 else '⚠️ '
    barra  = '█' * (n // 20)
    print(f'  {estado} {clase:<25} {n:>5} imágenes  {barra}')

## 2. Distribución de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Colores según el recipiente de cada clase
colores_map = {
    'lata':                  '#F1C40F',
    'botella_pet':           '#2E86C1',
    'botella_vidrio':        '#27AE60',
    'papel':                 '#7F8C8D',
    'carton':                '#95A5A6',
    'bolsa_plastica':        '#5DADE2',
    'tetrapak':              '#85C1E9',
    'organico':              '#8D6E63',
    'electronicos_pequenos': '#E74C3C',
    'no_reciclable':         '#2C3E50',
}
colores = [colores_map.get(c, '#AAB7B8') for c in CLASSES]

# Gráfica de barras
valores = [conteo[c] for c in CLASSES]
bars = axes[0].bar(range(len(CLASSES)), valores, color=colores, edgecolor='white', linewidth=0.8)
axes[0].set_xticks(range(len(CLASSES)))
axes[0].set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Distribución de imágenes por clase', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de imágenes')
axes[0].axhline(y=50, color='red', linestyle='--', linewidth=1, label='Mínimo requerido (50)')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=8, fontweight='bold')

# Gráfica de torta
no_vacias = [(c, v) for c, v in conteo.items() if v > 0]
if no_vacias:
    clases_pie, vals_pie = zip(*no_vacias)
    cols_pie = [colores_map.get(c, '#AAB7B8') for c in clases_pie]
    axes[1].pie(vals_pie, labels=clases_pie, colors=cols_pie,
                autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
    axes[1].set_title('Proporción por clase', fontsize=13, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'Sin datos aún\nEjecuta download_datasets.py',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)

plt.suptitle('EDA — Reciclaje Inteligente · Grupo #4 · UMG 2026',
             fontsize=11, color='gray', y=1.01)
plt.tight_layout()
plt.savefig('../docs/eda_distribucion_clases.png', bbox_inches='tight')
plt.show()
print('📊 Gráfica guardada en docs/eda_distribucion_clases.png')

## 3. Muestras visuales por clase

In [ ]:
N_MUESTRAS = 5  # imágenes por clase a mostrar

clases_con_datos = [c for c in CLASSES if conteo[c] > 0]

if not clases_con_datos:
    print('⚠️  No hay imágenes aún. Ejecuta download_datasets.py primero.')
else:
    fig, axes = plt.subplots(
        len(clases_con_datos), N_MUESTRAS,
        figsize=(N_MUESTRAS * 2.5, len(clases_con_datos) * 2.5)
    )
    if len(clases_con_datos) == 1:
        axes = [axes]

    for i, clase in enumerate(clases_con_datos):
        muestras = rutas[clase][:N_MUESTRAS]
        for j in range(N_MUESTRAS):
            ax = axes[i][j] if N_MUESTRAS > 1 else axes[i]
            ax.axis('off')
            if j < len(muestras):
                try:
                    img = Image.open(muestras[j]).convert('RGB').resize((150, 150))
                    ax.imshow(img)
                    if j == 0:
                        emoji = COLOR_MAP[clase][2] if clase in COLOR_MAP else ''
                        ax.set_ylabel(f'{emoji} {clase}', rotation=0, labelpad=60,
                                      va='center', fontsize=9, fontweight='bold')
                except Exception:
                    ax.text(0.5, 0.5, 'Error', ha='center', va='center',
                            transform=ax.transAxes, fontsize=8)

    plt.suptitle(f'Muestras visuales por clase ({N_MUESTRAS} por clase)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../docs/eda_muestras_por_clase.png', bbox_inches='tight')
    plt.show()
    print('📸 Muestras guardadas en docs/eda_muestras_por_clase.png')

## 4. Análisis de tamaño de imágenes

In [ ]:
print('📐 Analizando tamaños de imágenes...')
print('   (Esto puede tomar unos segundos)\n')

anchos, altos, relaciones = [], [], []
muy_pequenas = []

for clase in CLASSES:
    for ruta in rutas[clase]:
        try:
            with Image.open(ruta) as img:
                w, h = img.size
                anchos.append(w)
                altos.append(h)
                relaciones.append(w / h)
                if w < 100 or h < 100:
                    muy_pequenas.append((str(ruta), w, h))
        except Exception:
            pass

if anchos:
    print(f'  Ancho  — min: {min(anchos)}px | max: {max(anchos)}px | promedio: {np.mean(anchos):.0f}px')
    print(f'  Alto   — min: {min(altos)}px  | max: {max(altos)}px  | promedio: {np.mean(altos):.0f}px')
    print(f'  Relación aspecto promedio: {np.mean(relaciones):.2f}')
    print(f'\n  Imágenes menores a 100×100 px (candidatas a eliminar): {len(muy_pequenas)}')
    if muy_pequenas:
        for path, w, h in muy_pequenas[:5]:
            print(f'    → {os.path.basename(path)} ({w}×{h})')

    # Histograma de tamaños
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(anchos, bins=30, color='#2E86C1', alpha=0.8, edgecolor='white')
    axes[0].axvline(224, color='red', linestyle='--', linewidth=1.5, label='Resize destino (224px)')
    axes[0].set_title('Distribución de anchos de imagen')
    axes[0].set_xlabel('Píxeles')
    axes[0].legend()

    axes[1].hist(altos, bins=30, color='#27AE60', alpha=0.8, edgecolor='white')
    axes[1].axvline(224, color='red', linestyle='--', linewidth=1.5, label='Resize destino (224px)')
    axes[1].set_title('Distribución de altos de imagen')
    axes[1].set_xlabel('Píxeles')
    axes[1].legend()

    plt.suptitle('Análisis de tamaños — Dataset Reciclaje Inteligente',
                 fontsize=12, color='gray')
    plt.tight_layout()
    plt.savefig('../docs/eda_tamanos_imagenes.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠️  No hay imágenes para analizar aún.')

## 5. Detección de imágenes borrosas

In [ ]:
def es_borrosa(ruta_img: str, umbral: float = 100.0) -> tuple:
    """
    Detecta si una imagen es borrosa usando la varianza del Laplaciano.
    Un valor bajo indica imagen borrosa.
    """
    try:
        img = cv2.imread(str(ruta_img))
        if img is None:
            return True, 0.0
        gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        var  = cv2.Laplacian(gris, cv2.CV_64F).var()
        return var < umbral, round(var, 2)
    except Exception:
        return True, 0.0

print('🔍 Detectando imágenes borrosas (umbral Laplaciano < 100)...')
borrosas = []

for clase in CLASSES:
    for ruta in rutas[clase]:
        borrosa, score = es_borrosa(ruta)
        if borrosa:
            borrosas.append((clase, str(ruta), score))

print(f'\n  Imágenes potencialmente borrosas encontradas: {len(borrosas)}')
if borrosas:
    print('  Primeras 10 candidatas a revisar:')
    for clase, path, score in sorted(borrosas, key=lambda x: x[2])[:10]:
        print(f'    [{clase}] {os.path.basename(path)} — score: {score}')
    print('\n  ⚠️  Revisa manualmente estas imágenes y elimina las que estén muy borrosas.')
else:
    print('  ✅ No se detectaron imágenes borrosas significativas.')

## 6. Resumen y conclusiones del EDA

In [ ]:
print('=' * 60)
print('  RESUMEN DEL ANÁLISIS EXPLORATORIO — Grupo #4 · UMG 2026')
print('=' * 60)

print(f'\n📦 Total de imágenes en dataset: {total}')
print(f'🗂️  Clases definidas: {len(CLASSES)}')

# Clases con suficientes datos
ok     = [c for c in CLASSES if conteo.get(c, 0) >= 50]
faltan = [c for c in CLASSES if conteo.get(c, 0) < 50]

print(f'\n✅ Clases con ≥ 50 imágenes ({len(ok)}): {ok}')
print(f'⚠️  Clases con < 50 imágenes ({len(faltan)}): {faltan}')

if faltan:
    print('\n🎯 Acciones requeridas:')
    for c in faltan:
        faltantes = 50 - conteo.get(c, 0)
        print(f'   → Fotografiar al menos {faltantes} imágenes más de: {c}')

print('\n📋 Decisiones de preprocesamiento justificadas por el EDA:')
print('   • Resize a 224×224 px — requerido por MobileNetV2')
print('   • Normalización con media/std de ImageNet — Transfer Learning')
print('   • Augmentation: rotación ±15°, flip horizontal, cambio de brillo')
print('   • División: 70% train / 15% val / 15% test (stratified split)')
if faltan:
    print('   • CrossEntropyLoss con pesos por clase — dataset desbalanceado')

print('\n📁 Archivos generados por este notebook:')
for f in ['docs/eda_distribucion_clases.png',
          'docs/eda_muestras_por_clase.png',
          'docs/eda_tamanos_imagenes.png']:
    existe = '✅' if os.path.exists(os.path.join('..', f)) else '⏳ (pendiente)'
    print(f'   {existe} {f}')

print('\n✅ EDA completado. Siguiente paso: notebooks/02_train.ipynb')